## Week 3: Data Preprocessing

Team ds55 member: Yingxin Deng

This week tasks: 
1. Handle missing values (decide whether to drop, impute, or flag).
2. Convert categorical fields to numeric (encoding).
3. Normalize numerical features if needed.
4. Create train/test split: use the most recent month of available data as the test set, and the X months immediately preceding it as the training set. X is not fixed — treat the training window length as a tunable choice and experiment to determine the optimal value of X.
5. Deliverable: 02_preprocessing.ipynb + cleaned CSV.

In [2]:
import numpy as np
import pandas as pd

In [3]:
files = [
    "../CRMLSSold202505.csv",
    "../CRMLSSold202506.csv",
    "../CRMLSSold202507.csv",
    "../CRMLSSold202508.csv",
    "../CRMLSSold202509.csv",
    "../CRMLSSold202510.csv",
    "../CRMLSSold202511.csv",
    "../CRMLSSold202512.csv",
    "../CRMLSSold202601.csv",
    "../CRMLSSold202602.csv",
    "../CRMLSSold202603.csv",
    "../CRMLSSold202604.csv",
    "../CRMLSSold202605.csv"
]

df = pd.concat(
    [pd.read_csv(f) for f in files],
    ignore_index=True
)

missing_summary = pd.DataFrame({
    "missing_count": df.isnull().sum(),
    "missing_percent": df.isnull().mean() * 100
})

missing_summary = missing_summary.sort_values("missing_percent", ascending=False)
missing_summary.head(30)

/var/folders/tf/71q912hd3fzd4yz1qccgzpcw0000gn/T/ipykernel_896/1194053380.py:18: DtypeWarning: Columns (0: WaterfrontYN) have mixed types. Specify dtype option on import or set low_memory=False.
  [pd.read_csv(f) for f in files],
/var/folders/tf/71q912hd3fzd4yz1qccgzpcw0000gn/T/ipykernel_896/1194053380.py:18: DtypeWarning: Columns (0: WaterfrontYN, 1: PostalCode) have mixed types. Specify dtype option on import or set low_memory=False.
  [pd.read_csv(f) for f in files],


,missing_count,missing_percent
FireplacesTotal,281823,100.000000
MiddleOrJuniorSchoolDistrict,281823,100.000000
ElementarySchoolDistrict,281823,100.000000
CoveredSpaces,281823,100.000000
AboveGradeFinishedArea,281823,100.000000
WaterfrontYN,281665,99.943936
TaxYear,281661,99.942517
BusinessType,281122,99.751262
TaxAnnualAmount,280887,99.667877
BelowGradeFinishedArea,280404,99.496492


## Handle Missing Values

To reduce noise and remove features with almost no useful information, columns with more than 90% missing values were dropped. Since fewer than 1% of observations contain valid values for these features, they provide very limited predictive value while increasing data sparsity and model complexity.

In addition, columns containing personal names or agent information (e.g., first names, last names, and office names) were removed. These features are not directly related to property characteristics and are unlikely to contribute meaningfully to house price prediction. They also have high cardinality, which would unnecessarily increase the complexity of feature encoding without providing significant predictive benefits.

In [4]:
drop_cols = [
    "FireplacesTotal",
    "MiddleOrJuniorSchoolDistrict",
    "ElementarySchoolDistrict",
    "CoveredSpaces",
    "AboveGradeFinishedArea",
    "WaterfrontYN",
    "TaxYear",  
    "BusinessType",
    "TaxAnnualAmount",
    "BelowGradeFinishedArea",
    "BuilderName",
    "LotSizeDimensions",
    "CoBuyerAgentFirstName",
    "CoListAgentFirstName",
    "CoListAgentLastName",
    "CoListOfficeName",
]

df = df.drop(columns=drop_cols)

After removing columns with more than 99% missing values, the remaining features were further examined to determine an appropriate missing value handling strategy. Numerical columns were summarized using descriptive statistics (describe()), while categorical and binary variables were inspected separately to understand their distributions and missing patterns. This analysis helps determine whether missing values should be imputed using the mean, median, mode, a new category (e.g., "Unknown"), or inferred from the business meaning of the feature.

In [5]:
# Fill categorical features with "Unknown"
categorical_fill = [
    "ElementarySchool",
    "MiddleOrJuniorSchool",
    "HighSchool",
    "HighSchoolDistrict",
    "SubdivisionName",
    "Flooring",
    "AttachedGarageYN"
]

for col in categorical_fill:
    df[col] = df[col].fillna("Unknown")

In [6]:
# Fill numerical features with median
median_fill = [
    "BuildingAreaTotal",
    "MainLevelBedrooms",
    "AssociationFee"
]

for col in median_fill:
    df[col] = df[col].fillna(df[col].median())

In [7]:
# Fill discrete numerical features with mode
mode_fill = [
    "AssociationFeeFrequency",
    "Stories",
    "GarageSpaces"
]

for col in mode_fill:
    df[col] = df[col].fillna(df[col].mode()[0])

Categorical Feature Encoding

Machine learning models cannot work directly with text data, so the remaining categorical features need to be converted into numerical values. At first, I considered using one-hot encoding for all categorical variables. However, after checking the number of unique values in each column, I realized that some features, such as school districts and postal codes, contain hundreds or even thousands of categories. One-hot encoding these features would create an extremely large number of sparse columns and make the dataset unnecessarily complex.

Therefore, I decided to use different encoding strategies based on the cardinality of each feature. Binary variables were encoded as 0 and 1, low-cardinality categorical variables were one-hot encoded, and high-cardinality features were label encoded to preserve useful information while keeping the dataset manageable for later modeling.

In [8]:
df.select_dtypes(include=["object"]).columns.tolist()

/var/folders/tf/71q912hd3fzd4yz1qccgzpcw0000gn/T/ipykernel_896/392840347.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df.select_dtypes(include=["object"]).columns.tolist()


['BuyerAgentAOR',
 'ListAgentAOR',
 'Flooring',
 'ViewYN',
 'BasementYN',
 'PoolPrivateYN',
 'ListAgentEmail',
 'CloseDate',
 'ListAgentFirstName',
 'ListAgentLastName',
 'UnparsedAddress',
 'PropertyType',
 'ListOfficeName',
 'BuyerOfficeName',
 'ListAgentFullName',
 'BuyerAgentMlsId',
 'BuyerAgentFirstName',
 'BuyerAgentLastName',
 'AssociationFeeFrequency',
 'MLSAreaMajor',
 'CountyOrParish',
 'MlsStatus',
 'ElementarySchool',
 'AttachedGarageYN',
 'PropertySubType',
 'SubdivisionName',
 'BuyerOfficeAOR',
 'ListingId',
 'City',
 'ContractStatusChangeDate',
 'PurchaseContractDate',
 'ListingContractDate',
 'StateOrProvince',
 'MiddleOrJuniorSchool',
 'FireplaceYN',
 'HighSchool',
 'Levels',
 'NewConstructionYN',
 'HighSchoolDistrict',
 'PostalCode']

In [9]:
drop_more_cols = [
    "ListAgentEmail",
    "ListAgentFirstName",
    "ListAgentLastName",
    "ListAgentFullName",
    "BuyerAgentFirstName",
    "BuyerAgentLastName",
    "BuyerAgentMlsId",
    "UnparsedAddress",
    "ListOfficeName",
    "BuyerOfficeName",
    "BuyerAgentAOR",
    "ListAgentAOR",
    "ListingId",
    "ContractStatusChangeDate",
    "PurchaseContractDate",
    "ListingContractDate",
    "SubdivisionName"
]

df = df.drop(columns=[col for col in drop_more_cols if col in df.columns])

In [10]:
yn_cols = [
    "ViewYN",
    "BasementYN",
    "PoolPrivateYN",
    "AttachedGarageYN",
    "FireplaceYN",
    "NewConstructionYN"
]

for col in yn_cols:
    if col in df.columns:
        df[col] = df[col].map({
            True: 1,
            False: 0,
            "True": 1,
            "False": 0,
            "Y": 1,
            "N": 0
        })

In [11]:
one_hot_cols = [
    "Flooring",
    "PropertyType",
    "AssociationFeeFrequency",
    "MLSAreaMajor",
    "CountyOrParish",
    "MlsStatus",
    "PropertySubType",
    "City",
    "StateOrProvince",
    "Levels"
]

df = pd.get_dummies(
    df,
    columns=[col for col in one_hot_cols if col in df.columns],
    drop_first=True
)

In [12]:
high_cardinality_cols = [
    "ElementarySchool",
    "MiddleOrJuniorSchool",
    "HighSchool",
    "HighSchoolDistrict",
    "PostalCode",
    "SubdivisionName",
    "BuyerOfficeAOR"
]

for col in high_cardinality_cols:
    if col in df.columns:
        print(col, df[col].nunique())

ElementarySchool 1331
MiddleOrJuniorSchool 633
HighSchool 502
HighSchoolDistrict 436
PostalCode 2843
BuyerOfficeAOR 59


In [13]:
from sklearn.preprocessing import LabelEncoder

label_cols = [
    "ElementarySchool",
    "MiddleOrJuniorSchool",
    "HighSchool",
    "HighSchoolDistrict",
    "PostalCode",
    "BuyerOfficeAOR"
]

for col in label_cols:
    if col in df.columns:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))

In [14]:
df["CloseDate"] = pd.to_datetime(df["CloseDate"])

## Train-Test Split

Since the project requires the training window length to be tunable, I started with a six-month training window as a reasonable baseline. Different values of X (e.g., 3, 6, 9, and 12 months) will be evaluated in the modeling stage to determine which window provides the best predictive performance.

In [15]:
df["CloseDate"] = pd.to_datetime(df["CloseDate"])
df = df.sort_values("CloseDate").reset_index(drop=True)
df["YearMonth"] = df["CloseDate"].dt.to_period("M")

print("Available months:")
print(df["YearMonth"].value_counts().sort_index())

latest_month = df["YearMonth"].max()

# Initial training window length
train_window = 6

# Get the X months immediately before the test month
train_months = pd.period_range(
    end=latest_month - 1,
    periods=train_window,
    freq="M"
)

train_df = df[df["YearMonth"].isin(train_months)].copy()
test_df = df[df["YearMonth"] == latest_month].copy()

print("\nLatest month / Test month:", latest_month)

print("\nTraining months:")
print(sorted(train_df["YearMonth"].unique()))

print("\nTest month:")
print(sorted(test_df["YearMonth"].unique()))

print("\nTrain shape:", train_df.shape)
print("Test shape:", test_df.shape)

train_df = train_df.drop(columns=["YearMonth"])
test_df = test_df.drop(columns=["YearMonth"])

Available months:
YearMonth
2025-05    23154
2025-06    22883
2025-07    23646
2025-08    22972
2025-09    22443
2025-10    23233
2025-11    19088
2025-12    20538
2026-01    16487
2026-02    18124
2026-03    22583
2026-04    23412
2026-05    23260
Freq: M, Name: count, dtype: int64

Latest month / Test month: 2026-05

Training months:
[Period('2025-11', 'M'), Period('2025-12', 'M'), Period('2026-01', 'M'), Period('2026-02', 'M'), Period('2026-03', 'M'), Period('2026-04', 'M')]

Test month:
[Period('2026-05', 'M')]

Train shape: (120232, 2690)
Test shape: (23260, 2690)


## Feature Normalization

Before training the model, I checked whether normalization was necessary for the numerical features. I noticed that variables such as living area, lot size, and property prices are measured on very different scales. If left unchanged, features with much larger values could have a greater influence on some machine learning models.

To make the numerical features more comparable, I used StandardScaler to standardize them so that they have a mean of 0 and a standard deviation of 1. Binary variables encoded as 0 and 1 were not normalized, since they already represent categorical information rather than continuous numerical values.

In [16]:
# Separate features and target
X_train = train_df.drop(columns=["ClosePrice"])
y_train = train_df["ClosePrice"]

X_test = test_df.drop(columns=["ClosePrice"])
y_test = test_df["ClosePrice"]

In [17]:
from sklearn.preprocessing import StandardScaler

scale_cols = [
    "LivingArea",
    "LotSizeAcres",
    "ListPrice",
    "OriginalListPrice",
    "AssociationFee",
    "BuildingAreaTotal",
    "DaysOnMarket",
    "Latitude",
    "Longitude"
]

scaler = StandardScaler()

X_train[scale_cols] = scaler.fit_transform(X_train[scale_cols])

X_test[scale_cols] = scaler.transform(X_test[scale_cols])

In [18]:
X_train[scale_cols].describe().T[["mean", "std"]].head()

,mean,std
LivingArea,3.869601e-17,1.000004
LotSizeAcres,2.593624e-19,1.000005
ListPrice,1.988691e-17,1.000004
OriginalListPrice,2.370787e-18,1.000004
AssociationFee,2.174793e-17,1.000004


In [19]:
# Save cleaned dataset
df.to_csv(
    "../week3/cleaned_data.csv.gz",
    index=False,
    compression="gzip"
)